# Lab 3: Object Oriented Programming Part I

Welcome to Lab 3 of Programming Methods! In this class we will start exploring Object Oriented Programming (OOP). The Lab has **three** main exercises and **six** wrap-up questions in the end. There is one "optional stretch" exercise.

Create a **new branch** for this Lab (e.g.,`git checkout -b lab-03-oop1`).

Then, complete the following exercises in **this** Jupyter Notebook. 

Once you are done **commit your work** using the proper Git commands.

The **solutions** for this lab will be published on the time of the next Lab.

Good luck :) 



### From scripts to reusable API clients

In the last class your code:
- Fetched data from the World Bank REST API
- Sent a prompt to a local Ollama model and parsed the JSON reply
- Built a simple chat loop against `ollama.chat()`

That code has three recurring problems:

1. **No validation** — you trust that the JSON has the shape you expect. If a field is missing or renamed, your code fails.
2. **Repetition** — you check `response.status_code == 200` and call `.json()` in multiple places.
3. **No reuse** — the logic for "format a prompt" and "send a request" is tangled together inside one script, so you can't easily plug in a new API without rewriting everything.

Today we will fix all three using **Pydantic** (validation) and **class hierarchies** (reuse). Last class's code will not be thrown away, it will be *refactored*.

By the end of this notebook you should have:
- The Pydantic model **OllamaResponse**.
- The base class **APIClient** with the shared request logic.
- The subclass **OllamaClient**.
- The class **ChatSession** that wraps the chat loop from Lab 2 Exercise 3.

## Setup

Run this cell first.


In [11]:
import requests
from pydantic import BaseModel, ValidationError

---
## Exercise 1: Classes and Pydantic Models

In the previous lab, we interacted with Ollama directly using `requests`.

In this exercise, we will improve that code in two steps:

1. Encapsulate the interaction with Ollama inside a **class**.
2. Use **Pydantic** to describe and validate the response returned by the API.

### 1a. Create an OllamaClient class

Create a class called `OllamaClient` that represents a client for the local Ollama API.

The class should:

- Receive the model name when instantiated.
- Store the Ollama API URL as an attribute.
- Have a method called `generate(prompt)` that:
    - Receives a prompt as a string.
    - Sends a `POST` request to `http://localhost:11434/api/generate`.
    - Sends the model and prompt in the JSON body.
    - Sets `"stream": False`.
    - Checks that the request was successful.
    - Returns the JSON response as a Python dictionary.

Instantiate the class and use it to ask the model a simple question.

In [12]:
class OllamaClient:
    def __init__(
        self,
        model: str,
        url: str = "http://localhost:11434/api/generate",
    ):
        self.model = model
        self.url = url

    def generate(self, prompt: str) -> dict:
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
        }

        response = requests.post(self.url, json=payload)

        response.raise_for_status()

        return response.json()


ollama = OllamaClient(model="llama3.2:1b")

result = ollama.generate("What is an API?")

print(result)
print(result["response"])

{'model': 'llama3.2:1b', 'created_at': '2026-09-21T18:01:14.916898Z', 'response': "An API, or Application Programming Interface, is a set of defined rules that enables different software systems or applications to communicate with each other. It's like a messenger that allows different parts of an application to exchange data, functionality, or services with each other.\n\nThink of it like a phonebook. When you want to find someone's phone number, you don't need to know the person's name or address. Instead, you can use the phonebook (the phonebook API) to look up the number and get the information you need.\n\nIn the same way, when different applications or services need to work together, they use an API to communicate and exchange data. This allows them to:\n\n* Share data and functionality\n* Coordinate their actions\n* Use each other's services\n* Integrate with each other\n\nAPIs can be used to power various applications, such as:\n\n* Web applications (e.g., social media, e-comme

### 1b. OllamaResponse Pydantic model

Currently, `generate()` returns a regular Python dictionary.

Let's use Pydantic to describe the structure of the response.

Look at the JSON returned by Ollama and create a Pydantic model called
`OllamaResponse`.

At minimum, it should validate:

- **model**: Name of the model used.
- **response**: Text generated by the model.
- **done**: Whether generation has finished.

Then use `OllamaResponse` to validate the dictionary returned by your
`OllamaClient`.

Try accessing the generated text using:

`result.response`

instead of:

`result["response"]`

In [13]:
class OllamaResponse(BaseModel):
    model: str
    response: str
    done: bool


raw_result = ollama.generate("What is an API?")

result = OllamaResponse.model_validate(raw_result)

print(result)
print(result.response)
print(result.model)
print(result.done)

model='llama3.2:1b' response="An API, or Application Programming Interface, is a set of rules, protocols, and tools that allows different software systems to communicate with each other. It's like a messenger between different systems, enabling them to exchange data, services, or functionality.\n\nThink of an API as a virtual intermediary that allows different applications, services, or devices to interact with each other. When you make a request to an API, you're essentially asking the API to perform a specific action, such as retrieving data, sending a message, or executing a task.\n\nHere's an example to illustrate the concept:\n\nSuppose you have a website for booking flights, and you want to integrate it with a hotel booking system. The API is like a messenger that allows the two systems to communicate. When you click on a flight button on the website, you're sending a request to the API, which then retrieves flight information from the airline's database and passes it to the hote

### 1c. Integrate Pydantic into the class

Finally, modify `OllamaClient.generate()` so that it returns an
`OllamaResponse` object directly instead of a dictionary.

Add the appropriate return type annotation to the method.

In [14]:
class OllamaResponse(BaseModel):
    model: str
    response: str
    done: bool


raw_result = ollama.generate("What is an API?")

result = OllamaResponse.model_validate(raw_result)

print(result)
print(result.response)
print(result.model)
print(result.done)

model='llama3.2:1b' response="An API, or Application Programming Interface, is a set of rules and protocols that allows different applications, systems, or services to communicate and exchange data with each other. It's like a messenger that enables different systems to talk to each other, without needing to understand the details of how they work internally.\n\nThink of an API as a middleman that facilitates communication between two or more systems. It defines how data is sent and received, and allows different systems to interact with each other in a standardized way.\n\nHere's an example to illustrate this:\n\nLet's say you want to order food online. You can use an API provided by a restaurant (the system) to place your order. The API defines the rules and protocols for sending and receiving data, such as:\n\n* What information do you need to provide to complete the order?\n* How do you want to pay for the order?\n* What are the available menu items and their prices?\n* How do you 

### 1d. Break it on purpose

1. Pass in a dictionary that is missing a required field, or that has the wrong type for **done**. Wrap the call in a `ValidationError` and print `e.errors()`.
3. Pass in a dictionary that has an additional field. Does it raise an exception?

**Question to answer in a markdown cell below your code:** what does Pydantic's
error message tell you that a plain `KeyError` would not?


In [15]:
# Missing field --> raises
try:
    bad = OllamaResponse(model="llama3.2:1b", response="hi")  # missing `done`
except ValidationError as e:
    print(e.errors())


[{'type': 'missing', 'loc': ('done',), 'msg': 'Field required', 'input': {'model': 'llama3.2:1b', 'response': 'hi'}, 'url': 'https://errors.pydantic.dev/2.13/v/missing'}]


In [16]:
# Extra, unexpected field --> does NOT raise, just gets silently ignored
extra_field_response = OllamaResponse(
    model="llama3.2:1b",
    response="hi",
    done=True,
    context=[1, 2, 3],       # not in our model at all
    prompt_eval_count=17,    # neither is this
)
print(extra_field_response)  # context and prompt_eval_count are simply gone

model='llama3.2:1b' response='hi' done=True


**Answer:** a plain `KeyError` would only tell you that a lookup failed — a single, isolated failure at the exact line
where you happened to access that key, with no context about what else might
also be wrong. Pydantic's `ValidationError` instead reports *every* invalid
or missing field in one pass, names the exact field, explains what
went wrong, and even shows the offending input. This allows a
complete diagnostic in one shot rather than discovering problems one
`KeyError` at a time.


---
## Exercise 2: Base `APIClient` Class Hierarchy

Our `OllamaClient` already handles several responsibilities that are common when interacting with APIs:

- Store a `base_url`
- Send HTTP requests
- Check whether a request was successful
- Parse the JSON response

Instead of implementing these behaviors directly in `OllamaClient`, let's move the common functionality into a reusable **base class** called `APIClient`.

`OllamaClient` will then **inherit** from `APIClient` and add only the behavior that is specific to Ollama.

This will give us the following class hierarchy:

```text
APIClient
    │
    └── OllamaClient
```

The goal is to separate **general API functionality** from **Ollama-specific functionality**, while introducing inheritance and code reuse.

#### A note on `super().__init__()`

You'll see `super().__init__(...)` inside the `__init__` methods of the subclasses below. Here's what it does and why it matters.

When a class such as **OllamaClient** inherits from **APIClient**, it *could* define its own `__init__` and set `self.base_url` directly, duplicating what `APIClient.__init__` already does.

Instead, `super().__init__(base_url=...)` calls the **parent class's** `__init__` method from inside the child's `__init__`. This means the parent's setup logic can be defined once and reused by its subclasses.

```python

class APIClient:
    def __init__(self, base_url: str):
        self.base_url = base_url          # parent's setup logic

class OllamaClient(APIClient):
    def __init__(self):
        super().__init__(
            base_url="http://localhost:11434"
        )
```
Why not just write `self.base_url = "http://localhost:11434"` directly inside `OllamaClient.__init__` and skip super() entirely? Two reasons:

1. DRY (Don’t Repeat Yourself). If `APIClient.__init__` later grows - for example, if it starts setting up a requests.Session() or a default timeout - every subclass that uses `super().__init__()` benefits from that change automatically. Subclasses that bypass it would miss the new behavior.
2. It respects the base class’s responsibility. APIClient defines what every API client fundamentally needs, such as a `base_url`. Subclasses only need to provide the appropriate value rather than reimplementing the setup themselves.

super() isn’t limited to __init__. It can be used with other methods as well. It refers to the next class in Python’s method resolution order (MRO). With simple single inheritance like this, you can think of it as referring to the parent class.

You’ll often see this pattern when a subclass wants to reuse or extend behavior defined by its parent class rather than implementing everything again.

### 2a. Create the APIClient class

Fill in the base class below. `_request` should be the *only* place in your
entire notebook that calls `requests.get` / `requests.post` and checks
`status_code`.

In [17]:
# TODO: remove solution but keep main skeleton
class APIClient:
    """Base class for talking to a JSON HTTP API."""

    def __init__(self, base_url: str):
        self.base_url = base_url

    def _request(self, method: str, endpoint: str, **kwargs) -> dict:
        url = f"{self.base_url}{endpoint}"
        response = requests.request(method, url, **kwargs)

        if response.status_code != 200:
            raise RuntimeError(
                f"Request to {url} failed with status {response.status_code}: "
                f"{response.text}"
            )

        return response.json()

    def format_prompt(self, *args, **kwargs) -> str:
        raise NotImplementedError


### 2b. Create the `OllamaClient` subclass

Implement an `OllamaClient` subclass that wraps the Ollama API at `http://localhost:11434`.

The class should:

1. Inherit from `APIClient`.
2. Use `super().__init__(...)` to configure the Ollama `base_url`.
3. Store the name of the Ollama model to use.
4. Implement a `generate(prompt)` method that:
   - Receives a prompt as a string.
   - Creates the appropriate JSON payload with `model`, `prompt`, and `stream`.
   - Calls `self._request(...)` to send the request.
   - Validates the response using the `OllamaResponse` Pydantic model created in Exercise 1.
   - Returns the validated `OllamaResponse`.

Finally, instantiate `OllamaClient`, send a prompt of your choice, and print the generated response.

In [18]:
class OllamaClient(APIClient):
    def __init__(
        self,
        model: str = "llama3.2:1b",
    ):
        super().__init__(
            base_url="http://localhost:11434"
        )
        self.model = model

    def generate(self, prompt: str) -> OllamaResponse:
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
        }

        data = self._request(
            "POST",
            "/api/generate",
            json=payload,
        )

        return OllamaResponse.model_validate(data)


ollama = OllamaClient()

result = ollama.generate("Explain what inheritance is in Python.")

print(result.response)

Inheritance in Python

Inheritance is a fundamental concept in object-oriented programming (OOP) that allows one class to inherit the properties and behavior of another class. It's a way to create a new class based on an existing class, and it enables code reuse and facilitates the creation of hierarchies of classes.

### Class Hierarchy

Here's an example of a simple class hierarchy in Python:

```python
# Animal class
class Animal:
    def __init__(self, name, age):
        self.name = name
        self.age = age

    def eat(self):
        print(f"{self.name} is eating.")

# Mammal class
class Mammal(Animal):
    def __init__(self, name, age, type):
        super().__init__(name, age)
        self.type = type

    def sleep(self):
        print(f"{self.name} is sleeping.")

# Dog class
class Dog(Mammal):
    def __init__(self, name, age, type, breed):
        super().__init__(name, age, type)
        self.breed = breed

    def bark(self):
        print(f"{self.name} the {self.breed

> #### 📚 Further reading
> - Nelson, *Software Engineering for Data Scientists*, **Chapter 4 "Object-Oriented Programming and Functional Programming"** — the foundational chapter for classes, methods, attributes, and defining your own classes. Read this first if `super().__init__()` above still feels shaky.
> - Nelson, **Chapter 8 "Design and Refactoring"** — covers modular code, interfaces and contracts, and coupling. This is the chapter that argues *why* you refactor procedural scripts into classes in the first place.
> - Ramalho, *Fluent Python* (2nd ed.), **Chapter 14 "Inheritance: For Better or for Worse"** — covers `super()` in more depth, plus multiple inheritance and method resolution order. Ramalho is more skeptical of inheritance than Nelson is, and argues for favoring composition in many cases. Worth reading side by side with Nelson's Chapter 4. Also check the wrap-up question at the end of this notebook.

---
## Exercise 3: Handling Validation Errors

Our `OllamaClient.generate()` method now converts the API response into an `OllamaResponse` Pydantic object.

But what happens if the API returns data that does **not** match the structure we expect?

### 3a. Explore Pydantic validation

Create an invalid response dictionary. For example, remove one of the required fields from a valid Ollama response:

```python
invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}
```

Try to validate it using `OllamaResponse.model_validate(...)`.

1. What exception does Pydantic raise?
2. What information does the exception give you about the invalid data?

In [24]:
invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}

OllamaResponse.model_validate(
    invalid_response
)


ValidationError: 1 validation error for OllamaResponse
done
  Field required [type=missing, input_value={'model': 'llama3.2:1b', 'response': 'Hello!'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing

### 3b. Handle validation errors in `generate`

Modify the generate() method of your existing OllamaClient class so that it:

1. Sends the request using `self._request(...)`.
2. Validates the returned JSON using `OllamaResponse`.
3. Catches `ValidationError` if the response does not match the expected structure.
4. Raises a clearer error explaining that the Ollama response could not be validated.

Test your error handling using the invalid response from the previous exercise.

In [23]:
from pydantic import ValidationError

invalid_response = {
    "model": "llama3.2:1b",
    "response": "Hello!"
    # "done" is missing
}

try:
    OllamaResponse.model_validate(invalid_response)
except ValidationError as exc:
    print(exc)

1 validation error for OllamaResponse
done
  Field required [type=missing, input_value={'model': 'llama3.2:1b', 'response': 'Hello!'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing


In [ ]:
class OllamaClient(APIClient):
    def __init__(self, model: str = "llama3.2:1b"):
        super().__init__(
            base_url="http://localhost:11434"
        )
        self.model = model

    def generate(self, prompt: str) -> OllamaResponse:
        payload = {
            "model": self.model,
            "prompt": prompt,
            "stream": False,
        }

        data = self._request(
            "POST",
            "/api/generate",
            json=payload,
        )

        try:
            return OllamaResponse.model_validate(data)

        except ValidationError as exc:
            fields = [
                ".".join(str(part) for part in error["loc"])
                for error in exc.errors()
            ]

            raise ValueError(
                f"Invalid Ollama response. "
                f"Validation failed for: {', '.join(fields)}"
            ) from exc

### 3c. Put it together end to end

Using `OllamaClient`, reproduce the Ollama interaction from the previous lab in just a few lines:

1. Create an instance of `OllamaClient`.
2. Call `generate(...)` with a prompt of your choice.
3. Print the generated text using `.response`.

Compare this with the procedural version from the previous lab, where you had to manually:

- Define the URL and payload.
- Send the HTTP request.
- Check the status code.
- Parse the JSON response.
- Extract the generated text.

What responsibilities are now hidden behind the `OllamaClient` interface?

In [25]:
ollama_client = OllamaClient(model="llama3.2:1b")

result = ollama_client.generate(
    "Explain what an API is in one sentence."
)

print(result.response)

An API, or Application Programming Interface, is a set of rules and protocols that allows different applications, systems, or services to communicate and exchange data with each other seamlessly.


> #### 📚 Further reading
> - Nelson, *Software Engineering for Data Scientists*, **Chapter 11 "APIs"** — covers calling an API, HTTP methods and status codes, and building your own API with FastAPI. Good confirmatory reading for the request-handling logic inside `_request`, and a pointer to what "the other side" (building rather than consuming an API) looks like.

---
### Optional Stretch: ChatSession

Refactor Lab 2 Exercise 3 chat loop into a class that wraps an `OllamaClient`. Import any missing packages.

Design goals:
- `messages` should become an **instance attribute**, and not a loose global list.
- A `send(user_input: str) -> str` method appends the user message, calls the
  model, appends and returns the assistant's reply.
- The `while True` input loop becomes a `run()` method that calls
  `send` and prints the reply.


In [21]:
import ollama

class ChatSession:
    def __init__(self, client: OllamaClient):
        self.client = client
        self.messages: list[dict] = []

    def send(self, user_input: str) -> str:
        self.messages.append({"role": "user", "content": user_input})

        response = ollama.chat(model=self.client.model, messages=self.messages)
        reply = response["message"]["content"]

        self.messages.append({"role": "assistant", "content": reply})
        return reply

    def run(self):
        print("Chat started. Type 'exit' or 'quit' to stop.")
        while True:
            user_input = input("You: ")
            if user_input.lower() in ("exit", "quit"):
                break
            reply = self.send(user_input)
            print(f"Bot: {reply}")


---

## Wrap-up questions (answer briefly in markdown)

**1. Which functionality is most appropriate to keep in the base `APIClient` class?**

A) Building Ollama prompts and choosing which model to use  
B) Sending HTTP requests, checking status codes, and parsing JSON responses  
C) Defining the fields of `OllamaResponse`  
D) Deciding what prompt the user should send  

**Answer:** B

The base class should contain behavior that can be reused by different API
clients. Sending requests, checking status codes, and parsing JSON are general
API operations, while Ollama-specific behavior belongs in `OllamaClient`.


**2. What is the main advantage of validating an API response with Pydantic?**

A) It makes the HTTP request faster  
B) It automatically fixes errors returned by the API  
C) It checks that the response has the expected structure and types as soon as
it is received  
D) It converts every API response into a string  

**Answer:** C

Pydantic validates the response when the model is constructed. For example,
if a required field such as `done` is missing, the problem is detected
immediately rather than causing an error later in the program.


**3. Suppose we want to add a client for another JSON-based API. What would be the most appropriate approach with our current design?**

A) Add all the new API-specific methods directly to `APIClient`  
B) Copy the complete `OllamaClient` class and change its URL  
C) Create another subclass of `APIClient` containing the behavior specific to
the new API  
D) Modify `_request()` every time we add a new API  

**Answer:** C

The new client can inherit the shared request functionality from `APIClient`
while implementing its own endpoints, payloads, and response models.


**4. What does `super().__init__(...)` do inside `OllamaClient`?**

A) Creates a second `OllamaClient` object  
B) Calls the parent class's `__init__` method  
C) Sends a request to the API  
D) Automatically initializes every attribute in `OllamaClient`  

**Answer:** B

`super().__init__(...)` allows `OllamaClient` to reuse the initialization
logic defined by `APIClient`, such as storing the `base_url`.


**5. Why does `OllamaClient` inherit from `APIClient`?**

A) Because `OllamaClient` needs to reuse general API request functionality  
B) Because Pydantic requires all API clients to use inheritance  
C) Because `requests.post()` only works inside a parent class  
D) Because every Python class must inherit from a custom class  

**Answer:** A

Inheritance allows `OllamaClient` to reuse the general functionality provided
by `APIClient` while adding behavior that is specific to Ollama.


**6. Which statement best describes the inheritance vs. composition trade-off for this example?**

A) Inheritance is always better because it requires less code  
B) Composition is always better because inheritance should never be used  
C) Inheritance is reasonable because `OllamaClient` is a specialized API
client, while composition could provide more flexibility for replacing
components such as the request handler  
D) There is no meaningful difference between inheritance and composition  

**Answer:** C

Inheritance fits this example because `OllamaClient` is a specialized version
of an API client and can reuse its common behavior. Composition could instead
give the client another object responsible for making requests, which can
provide more flexibility, particularly when replacing or mocking that
behavior.